# HiSparse 入门 (1/8)：为什么需要它 —— 长上下文 decode 的带宽墙

> **一句话结论**：HiSparse 是一种长上下文 KV 卸载方案。它不把全部历史 KV 塞进设备 HBM，而是每一步只把当前 top-k 命中的那几行搬进设备热区，其余历史留在 Host 侧。

## 1. 先理解问题：decode 一步在干什么

Transformer 生成文本时，每一步只新算 **1 个 token**，但注意力要回看**整条历史序列**：

- 序列越长，每步要读取的 KV 行越多；
- 设备 HBM 容量有限，多请求、长序列放不下全部历史；
- 于是瓶颈往往不是算力，而是 **KV 读取带宽**（数据搬不动）。

## 2. 一个生活比喻

把历史 KV 想象成一座图书馆：

- **Host 池** = 整座图书馆（容量大，但取书慢）；
- **Resident 池** = 你个人书架（已经搬回家的常用书）；
- **Hot 池** = 书桌桌面（当前这一步正在翻的几本书）。

每一步只需要把"当前要翻的书"放到桌面，不必把整座图书馆搬上来。HiSparse 做的就是这套"挑书—搬运—放回"的索引与搬移管理。

## 3. 它解决什么，不解决什么

- **解决**：长上下文下 KV 驻留容量与读取带宽问题；把缺失历史按需搬入设备，注意力在设备热区原地读取。
- **不改变**：注意力的数学公式本身；算子只负责索引、搬移、组装与失效，注意力算术沿用既有稀疏 Flash Attention（SFA）。

## 4. 在 vllm-ascend 里的位置

这套能力分成两层：

- **vLLM Core**：持有 HMA 分配、Coordinator 准入/淘汰、请求生命周期、分组预取、Host 镜像完成与内存预算口径；
- **Ascend 后端**：供应驻留解析、Host 传输映射、共享 slab 视图适配、缓存算子与流/事件围栏。

## 5. 小结

- 长上下文 decode 的主要压力是 **KV 读取带宽与 HBM 容量**；
- HiSparse 用三级存储 + 按需搬移，让设备只保留当前命中的热行；
- 下一篇讲清这三级池的职责边界与统一行格式。

**下一篇**：`02-three-tier-cache.ipynb`
